# 05_full_cv_experiments.ipynb

## Objetivo

Ejecutar el experimento principal del trabajo terminal usando:

- validación cruzada sobre todo el dataset
- comparación de modelos
- comparación de técnicas de prompting
- análisis incremental de reglas de simplificación

## Configuración experimental

### Modelos
- llama3
- mistral

### Técnicas
- zero-shot
- few-shot

### Rulesets
- R0
- R1
- R2
- R3
- R4

## Esquema de partición

Dataset total
- 1 fold → TEST
- 9 folds → desarrollo
  - 75% → TRAIN
  - 25% → VALIDATION

## Consideración importante

Los ejemplos usados para few-shot se eliminan del dataset antes de crear los folds, para evitar fuga de información.

## Lógica metodológica

Este notebook corresponde a la fase experimental formal del proyecto.

Aquí se busca responder principalmente:

1. si few-shot mejora respecto a zero-shot
2. si las reglas progresivas R0–R4 aportan mejoras reales
3. cómo cambia el comportamiento según el modelo
4. qué configuración ofrece mejor equilibrio entre fidelidad, claridad y legibilidad

A diferencia de los notebooks anteriores, aquí ya se trabaja con:

- todo el dataset
- folds reproducibles
- train / validation / test
- logging estructurado por fold y por split

In [1]:
import sys
import os
import re
import gc
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("configs exists:", (PROJECT_ROOT / "configs").exists())
print("src exists:", (PROJECT_ROOT / "src").exists())

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
configs exists: True
src exists: True


In [3]:
from configs.models import MODELS, DEFAULT_GENERATION_CONFIG
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.utils.logging_utils import ExperimentLogger

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# =========================
# Configuración principal
# =========================
EXPERIMENT_ID = "exp_full_cv_v1"
DATASET_NAME = "FEINA"
RANDOM_SEED = 42
N_SPLITS = 10

SELECTED_MODELS = ["llama3", "mistral"]
SELECTED_PROMPT_TYPES = ["zero-shot", "few-shot"]
SELECTED_RULESETS = ["R0", "R1", "R2", "R3", "R4"]

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CV_DIR = OUTPUTS_DIR / "cv_runs"
LOGS_DIR = OUTPUTS_DIR / "logs"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
CV_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("CV_DIR:", CV_DIR)
print("LOGS_DIR:", LOGS_DIR)

EXPERIMENT_ID: exp_full_cv_v1
CV_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/cv_runs
LOGS_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/logs


In [5]:
# =========================
# Few-shot examples fijos
# =========================
TEST_SET = {
    "meta": {
        "name": "feina_manual_test_v1",
        "language": "es",
        "notes": "Textos difíciles/diversos"
    },
    "items": [
        {
            "id": "t1_largo",
            "texto": "La clasificación y las tasas de la ley de renta vigente durante 2019 se resumen a continuación: categorías del impuesto a la renta categorías tipos de rentas tasas de impuesto afecta a primera categoría rentas del capital tasa general de 25 % o 27,5 % las utilidades de empresas e inversiones de personas segunda categoría rentas del trabajo progresiva por tramos(desde 0 % a 40 %)los ingresos de trabajadores dependientes y pensionados impuesto global complementario suma de todas las rentas progresiva por tramos (desde 0 % a 40 %),menos los impuestos de primera y segunda categorías pagados todas las rentas (del capital y del trabajo) de las personas impuesto adicional rentas de no residentes (extranjeros)tasa de 35 % las transferencias fuera del país.",
            "gold": "A continuación, resumimos la clasificación y las tasas de la Ley de Renta vigente en dos mil diecinueve: Categorías del impuesto a la renta.                        Primera categoría tiene una tasa general de veinticinco o veintisiete coma cinco por ciento de las ganancias de empresas e inversiones de personas. Segunda categoría son las rentas del trabajo progresivo por tramos desde cero a cuarenta por ciento. Los ingresos de trabajadores dependientes y pensionados.                                              El impuesto global complementario es la suma de todas las rentas progresiva por tramos menos los impuestos de primera y segunda categorías pagados.  El impuesto adicional de rentas de extranjeros tiene una tasa de treinta y cinco por ciento las transferencias fuera del país."
        },
        {
            "id": "t2_numeros",
            "texto": "Las posiciones dadas por los autores mencionados se han trasladado en el tiempo a nuestros días y se han constituido, guardada la situación de la actualidad, en focos orientadores para entender que los impuestos son algo más que declaraciones de cobro o una simple exigencia impositiva y necesaria para la vida del estado y de las comunidades nacionales sin que medie nada más, ya que los impuestos que no son claros, equitativos, justos e imparciales para con todos los contribuyentes pierden toda su fuerza como obligación moral, y no son indicadores para el sostenimiento de una legítima democracia y esto porque si Las cargas impositivas no son proporcionales para todos, no existe entonces, -sostiene de Tocqueville- motivo, razón e interés para que la gente se aproxime, reúna y discuta.",
            "gold": "Los planteamientos de los autores mencionados se trasladan al tiempo actual y se estudian como ideas orientadoras para entender que los impuestos no son solo declaraciones de cobro o una simple exigencia impositiva y necesaria para la vida del estado y de las comunidades nacionales. Los impuestos que no son claros, equitativos, justos e imparciales para todos los contribuyentes pierden toda su fuerza como obligación moral y no son indicadores para el sostenimiento de una legítima democracia. Al respecto Tocqueville afirma que, si las cargas impositivas no son proporcionales para todos, no existe motivo, razón e interés para que la gente se aproxime, reúna y discuta."
        },
        {
            "id": "t3_formato_raro",
            "texto": "Numerosas personas en el mundo y en nuestro país afrontan problemas como éste, lo cual es muy deprimente porque además de una carga social costosa para el Estado, lo es más aún para las familias o parientes, que ante la indiferencia o subadministración de aquel para responder por tal servicio se ven compelidas a asumirlo por sí mism",
            "gold": "Muchas personas afrontan problemas como este. Esto es deprimente porque significa una carga social costosa para el Estado y sobre todo para las familias. Pues deben asumir este servicio por la indiferencia de la persona para hacerlo y es más problemático cuando no hay recursos."
        }
    ]
}

few_shot_examples = [
    {"source": item["texto"], "target": item["gold"]}
    for item in TEST_SET["items"]
]

few_shot_example_ids = [item["id"] for item in TEST_SET["items"]]

print("Few-shot examples:", len(few_shot_examples))
print("Few-shot example ids:", few_shot_example_ids)

Few-shot examples: 3
Few-shot example ids: ['t1_largo', 't2_numeros', 't3_formato_raro']


## Carga del dataset FEINA

Aquí se carga el corpus principal y se normalizan las columnas para dejar una estructura estándar:

- sample_id
- source_text
- reference_text

In [7]:
DATA_DIR = PROJECT_ROOT / "data"

CANDIDATE_DATASETS = [
    DATA_DIR / "Dataset_FEINA.xlsx",
]

dataset_path = None
for p in CANDIDATE_DATASETS:
    if p.exists():
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError("No se encontró el dataset FEINA en data/.")

print("Dataset encontrado:", dataset_path)

Dataset encontrado: /home/harielpadillasanchez/Documentos/TT/TT2/data/Dataset_FEINA.xlsx


In [8]:
if dataset_path.suffix.lower() == ".csv":
    df_raw = pd.read_csv(dataset_path)
elif dataset_path.suffix.lower() in [".xlsx", ".xls"]:
    df_raw = pd.read_excel(dataset_path)
else:
    raise ValueError(f"Formato no soportado: {dataset_path.suffix}")

print("Shape original:", df_raw.shape)
display(df_raw.head(3))

Shape original: (5313, 15)


,Unnamed: 0,idFinal,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,0,58_LibroBAC.pdf,"Como antes explicamos, las finanzas y los conc...","Como antes explicamos, las finanzas están sust...",Fiorella,15.0,21.0,8.0,5.0,NaN,NaN,NaN,NaN,NaN,0
1,1,60_LibroBAC.pdf,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos asuntos, se abordan al...",Fiorella,1.0,5.0,15.0,17.0,2.0,19.0,NaN,NaN,NaN,1
2,2,62_LibroBAC.pdf,"Pero aquí no termina la utilidad del libro, ya...","Pero aquí no termina la utilidad del libro, pu...",Fiorella,15.0,19.0,5.0,16.0,6.0,NaN,NaN,NaN,NaN,0


In [9]:
def normalize_col(col: str) -> str:
    col = str(col).strip().lower()
    col = re.sub(r"\s+", "_", col)
    col = col.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")
    col = col.replace("ñ", "n")
    return col

df = df_raw.copy()
df.columns = [normalize_col(c) for c in df.columns]

print("Columnas normalizadas:")
print(df.columns.tolist())

Columnas normalizadas:
['unnamed:_0', 'idfinal', 'segmento', 'propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


In [10]:
possible_id_cols = ["idfinal", "id", "segment_id", "sentence_id"]
possible_source_cols = ["segmento", "segment", "texto", "source", "input_text"]
possible_target_cols = ["propuesta", "proposal", "texto_simplificado", "target", "reference"]

def first_existing(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

ID_COL = first_existing(possible_id_cols, df.columns)
SOURCE_COL = first_existing(possible_source_cols, df.columns)
TARGET_COL = first_existing(possible_target_cols, df.columns)

print("ID_COL:", ID_COL)
print("SOURCE_COL:", SOURCE_COL)
print("TARGET_COL:", TARGET_COL)

if SOURCE_COL is None:
    raise ValueError("No se encontró la columna del texto original.")

ID_COL: idfinal
SOURCE_COL: segmento
TARGET_COL: propuesta


In [11]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else None

work_df = df.copy()
work_df["source_text"] = work_df[SOURCE_COL].apply(clean_text)
work_df["reference_text"] = work_df[TARGET_COL].apply(clean_text) if TARGET_COL else None

if ID_COL is not None:
    work_df["sample_id"] = work_df[ID_COL].astype(str)
else:
    work_df["sample_id"] = [f"sample_{i}" for i in range(len(work_df))]

work_df = work_df.dropna(subset=["source_text"]).copy()
work_df = work_df.drop_duplicates(subset=["source_text"]).reset_index(drop=True)

work_df = work_df[["sample_id", "source_text", "reference_text"]].copy()

print("Shape limpio:", work_df.shape)
display(work_df.head(5))

Shape limpio: (5298, 3)


,sample_id,source_text,reference_text
0,58_LibroBAC.pdf,"Como antes explicamos, las finanzas y los conc...","Como antes explicamos, las finanzas están sust..."
1,60_LibroBAC.pdf,"Una vez dirimidos estos asuntos, se entra de l...","Una vez resueltos estos asuntos, se abordan al..."
2,62_LibroBAC.pdf,"Pero aquí no termina la utilidad del libro, ya...","Pero aquí no termina la utilidad del libro, pu..."
3,63_LibroBAC.pdf,"Por lo demás, el libro en sí, termina siendo ú...","Por lo demás, el libro es útil para personas p..."
4,69_LibroBAC.pdf,Servir de documento de lectura y consulta sobr...,Debe servir como documento de lectura y consul...


## Eliminación de textos usados para few-shot

Para evitar fuga de información, se eliminan del dataset todos los textos que coinciden con los ejemplos usados en el prompt few-shot.

In [12]:
few_shot_source_texts = {re.sub(r"\s+", " ", x["texto"]).strip() for x in TEST_SET["items"]}

before = len(work_df)
work_df = work_df[~work_df["source_text"].isin(few_shot_source_texts)].copy().reset_index(drop=True)
after = len(work_df)

print("Antes:", before)
print("Después de quitar few-shot texts:", after)
print("Eliminados:", before - after)

Antes: 5298
Después de quitar few-shot texts: 5296
Eliminados: 2


## Generación de folds

Se usa validación cruzada externa con 10 folds.

En cada iteración:
- 1 fold = test
- los otros 9 folds = desarrollo
- dentro del conjunto de desarrollo se hace:
  - 75% train
  - 25% validation

In [13]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

fold_splits = []

for fold_id, (dev_idx, test_idx) in enumerate(kf.split(work_df), start=0):
    dev_df = work_df.iloc[dev_idx].copy().reset_index(drop=True)
    test_df = work_df.iloc[test_idx].copy().reset_index(drop=True)

    train_df, val_df = train_test_split(
        dev_df,
        test_size=0.25,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    fold_splits.append({
        "fold_id": fold_id,
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
    })

print("Folds creados:", len(fold_splits))
for f in fold_splits[:2]:
    print(
        f"fold={f['fold_id']} | train={len(f['train_df'])} | "
        f"val={len(f['val_df'])} | test={len(f['test_df'])}"
    )

Folds creados: 10
fold=0 | train=3574 | val=1192 | test=530
fold=1 | train=3574 | val=1192 | test=530


In [14]:
# Verificación rápida
total_test = sum(len(f["test_df"]) for f in fold_splits)
print("Total de ejemplos cubiertos por test en todos los folds:", total_test)
print("Tamaño dataset limpio:", len(work_df))

Total de ejemplos cubiertos por test en todos los folds: 5296
Tamaño dataset limpio: 5296


## Construcción de la matriz experimental

Cada texto de cada split se combina con:

- 2 modelos
- 2 técnicas
- 5 rulesets

lo que da:

20 configuraciones por texto

In [15]:
def build_experiment_rows(df_split, fold_id, split_name):
    rows = []

    for _, row in df_split.iterrows():
        for model_name in SELECTED_MODELS:
            for prompt_type in SELECTED_PROMPT_TYPES:
                for ruleset_name in SELECTED_RULESETS:
                    technique_name = f"{'ZS' if prompt_type == 'zero-shot' else 'FS'}_{ruleset_name}"

                    rows.append({
                        "fold_id": fold_id,
                        "split_name": split_name,
                        "sample_id": row["sample_id"],
                        "source_text": row["source_text"],
                        "reference_text": row["reference_text"],
                        "model_name": model_name,
                        "prompt_type": prompt_type,
                        "ruleset_name": ruleset_name,
                        "technique_name": technique_name,
                    })

    return pd.DataFrame(rows)

In [16]:
# Vista de ejemplo con el fold 0
fold0 = fold_splits[0]

train_exp_example = build_experiment_rows(fold0["train_df"], fold_id=0, split_name="train")
val_exp_example = build_experiment_rows(fold0["val_df"], fold_id=0, split_name="validation")
test_exp_example = build_experiment_rows(fold0["test_df"], fold_id=0, split_name="test")

print("Train example shape:", train_exp_example.shape)
print("Val example shape:", val_exp_example.shape)
print("Test example shape:", test_exp_example.shape)

display(test_exp_example.head(5))

Train example shape: (71480, 9)
Val example shape: (23840, 9)
Test example shape: (10600, 9)


,fold_id,split_name,sample_id,source_text,reference_text,model_name,prompt_type,ruleset_name,technique_name
0,0,test,77_LibroBAC.pdf,Para la elaboración del documento se integró u...,Se integró un equipo profesional interdiscipli...,llama3,zero-shot,R0,ZS_R0
1,0,test,77_LibroBAC.pdf,Para la elaboración del documento se integró u...,Se integró un equipo profesional interdiscipli...,llama3,zero-shot,R1,ZS_R1
2,0,test,77_LibroBAC.pdf,Para la elaboración del documento se integró u...,Se integró un equipo profesional interdiscipli...,llama3,zero-shot,R2,ZS_R2
3,0,test,77_LibroBAC.pdf,Para la elaboración del documento se integró u...,Se integró un equipo profesional interdiscipli...,llama3,zero-shot,R3,ZS_R3
4,0,test,77_LibroBAC.pdf,Para la elaboración del documento se integró u...,Se integró un equipo profesional interdiscipli...,llama3,zero-shot,R4,ZS_R4


In [17]:
logger = ExperimentLogger(base_dir="outputs/logs")

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "run_ts": RUN_TS,
    "dataset_name": DATASET_NAME,
    "models": SELECTED_MODELS,
    "prompt_types": SELECTED_PROMPT_TYPES,
    "rulesets": SELECTED_RULESETS,
    "n_splits": N_SPLITS,
    "few_shot_example_ids": few_shot_example_ids,
    "default_generation_config": DEFAULT_GENERATION_CONFIG,
    "notes": [
        "Experimento principal con cross validation",
        "Comparación de zero-shot y few-shot",
        "Evaluación incremental de rulesets R0-R4",
        "Se excluyen los ejemplos few-shot del dataset experimental",
    ],
}

logger.save_manifest(EXPERIMENT_ID, manifest)
print("Manifest guardado.")

Manifest guardado.


In [18]:
runner = ExperimentRunner(experiment_id=EXPERIMENT_ID)
print("Runner listo.")

Runner listo.


## Ejecución experimental

Se guardan resultados por separado para:

- train
- validation
- test

Además, cada corrida conserva:

- fold_id
- split_name
- técnica
- ruleset
- modelo
- ids de ejemplos few-shot usados

In [19]:
all_train_records = []
all_val_records = []
all_test_records = []

In [20]:
def record_to_row(record, technique_name):
    return {
        "experiment_id": record.experiment_id,
        "run_ts": RUN_TS,
        "dataset_name": record.dataset_name,
        "fold_id": record.fold_id,
        "split_name": record.split_name,
        "sample_id": record.sample_id,
        "model_name": record.model_key,
        "model_id": record.model_id,
        "backend": record.backend,
        "prompt_type": record.prompt_type,
        "ruleset_name": record.ruleset,
        "technique_name": technique_name,
        "few_shot_example_ids": record.few_shot_example_ids,
        "source_text": record.source_text,
        "reference_text": record.reference_text,
        "generated_text": record.generated_text,
        "prompt_text": record.prompt_text,
        "status": record.status,
        "error_message": record.error_message,
        "inference_seconds": record.inference_seconds,
        "generation_config": json.dumps(record.generation_config, ensure_ascii=False),
    }

In [21]:
def run_split_experiment(exp_df, split_name, fold_id):
    split_records = []

    print(f"\n===== FOLD {fold_id} | SPLIT {split_name.upper()} =====")
    print("Corridas esperadas:", len(exp_df))

    for i, row in exp_df.iterrows():
        print(
            f"[fold={fold_id} | {split_name} | {i+1}/{len(exp_df)}] "
            f"sample={row.sample_id} | model={row.model_name} | "
            f"prompt={row.prompt_type} | ruleset={row.ruleset_name}"
        )

        if row.prompt_type == "few-shot":
            current_few_shot_examples = few_shot_examples
            current_few_shot_example_ids = few_shot_example_ids
        else:
            current_few_shot_examples = None
            current_few_shot_example_ids = None

        record = runner.run_one(
            dataset_name=DATASET_NAME,
            model_key=row.model_name,
            prompt_type=row.prompt_type,
            ruleset=row.ruleset_name,
            source_text=row.source_text,
            reference_text=row.reference_text,
            sample_id=row.sample_id,
            fold_id=fold_id,
            split_name=split_name,
            few_shot_examples=current_few_shot_examples,
            few_shot_example_ids=current_few_shot_example_ids,
            generation_config=DEFAULT_GENERATION_CONFIG,
        )

        split_records.append(record_to_row(record, row.technique_name))

        gc.collect()

    return pd.DataFrame(split_records)

## Corrida completa fold por fold

Esta sección puede tardar bastante, porque es la corrida formal del experimento.

In [22]:
for fold_pack in fold_splits:
    fold_id = fold_pack["fold_id"]

    train_exp_df = build_experiment_rows(fold_pack["train_df"], fold_id=fold_id, split_name="train")
    val_exp_df = build_experiment_rows(fold_pack["val_df"], fold_id=fold_id, split_name="validation")
    test_exp_df = build_experiment_rows(fold_pack["test_df"], fold_id=fold_id, split_name="test")

    train_results_df = run_split_experiment(train_exp_df, split_name="train", fold_id=fold_id)
    val_results_df = run_split_experiment(val_exp_df, split_name="validation", fold_id=fold_id)
    test_results_df = run_split_experiment(test_exp_df, split_name="test", fold_id=fold_id)

    all_train_records.append(train_results_df)
    all_val_records.append(val_results_df)
    all_test_records.append(test_results_df)

    # Guardado parcial por fold
    fold_dir = CV_DIR / f"fold_{fold_id:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_results_df.to_csv(fold_dir / "train_results.csv", index=False, encoding="utf-8")
    val_results_df.to_csv(fold_dir / "validation_results.csv", index=False, encoding="utf-8")
    test_results_df.to_csv(fold_dir / "test_results.csv", index=False, encoding="utf-8")

    print(f"\nFold {fold_id} guardado en {fold_dir}")

    runner.full_cleanup()


===== FOLD 0 | SPLIT TRAIN =====
Corridas esperadas: 71480
[fold=0 | train | 1/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=zero-shot | ruleset=R0
[fold=0 | train | 2/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=zero-shot | ruleset=R1
[fold=0 | train | 3/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=zero-shot | ruleset=R2
[fold=0 | train | 4/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=zero-shot | ruleset=R3
[fold=0 | train | 5/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=zero-shot | ruleset=R4
[fold=0 | train | 6/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=few-shot | ruleset=R0
[fold=0 | train | 7/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=few-shot | ruleset=R1
[fold=0 | train | 8/71480] sample=473_LibroNEFE_Sincopyright.pdf | model=llama3 | prompt=few-shot | ruleset=R2
[fold=0 | train | 9/71480] sample=473_LibroNEFE

KeyboardInterrupt: 

In [ ]:
train_all_df = pd.concat(all_train_records, ignore_index=True) if all_train_records else pd.DataFrame()
val_all_df = pd.concat(all_val_records, ignore_index=True) if all_val_records else pd.DataFrame()
test_all_df = pd.concat(all_test_records, ignore_index=True) if all_test_records else pd.DataFrame()

print("Train total:", train_all_df.shape)
print("Validation total:", val_all_df.shape)
print("Test total:", test_all_df.shape)

In [ ]:
display(train_all_df.head(2))
display(val_all_df.head(2))
display(test_all_df.head(2))

## Export global

In [ ]:
TRAIN_OUT = CV_DIR / "all_train_results.csv"
VAL_OUT = CV_DIR / "all_validation_results.csv"
TEST_OUT = CV_DIR / "all_test_results.csv"
MANIFEST_OUT = CV_DIR / "manifest_cv.json"

train_all_df.to_csv(TRAIN_OUT, index=False, encoding="utf-8")
val_all_df.to_csv(VAL_OUT, index=False, encoding="utf-8")
test_all_df.to_csv(TEST_OUT, index=False, encoding="utf-8")

with open(MANIFEST_OUT, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("TRAIN_OUT:", TRAIN_OUT)
print("VAL_OUT:", VAL_OUT)
print("TEST_OUT:", TEST_OUT)
print("MANIFEST_OUT:", MANIFEST_OUT)

In [ ]:
print("Status train:")
display(train_all_df["status"].value_counts(dropna=False))

print("Status validation:")
display(val_all_df["status"].value_counts(dropna=False))

print("Status test:")
display(test_all_df["status"].value_counts(dropna=False))

In [ ]:
summary_test = test_all_df.groupby(
    ["model_name", "prompt_type", "ruleset_name", "technique_name"]
).agg(
    runs=("sample_id", "size"),
    success_rate=("status", lambda s: (s == "success").mean()),
    mean_time=("inference_seconds", "mean"),
).reset_index()

display(summary_test.head(20))

## Cierre metodológico

Al terminar este notebook, ya se tendrá la corrida formal del experimento sobre todo el dataset.

Los archivos generados servirán como entrada para el siguiente notebook:

`06_full_cv_analysis.ipynb`

En ese notebook se calcularán las métricas automáticas y se analizarán formalmente:

- zero-shot vs few-shot
- efecto incremental de R0 → R4
- comparación entre modelos
- resultados finales para tablas del TT

In [ ]:
runner.full_cleanup()
print("Limpieza final completada.")